# eCommerce Funnel Analysis
**Goal:** Calculate unique user conversion rates from View -> Intent -> Purchase.

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option('display.max_columns', None)

In [10]:
df = pd.read_csv("../data/raw/2019-Oct.csv", nrows= 500000)
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


## Step 1: Raw Event Frequencies
First, looking at the total number of actions taken across the platform. However, these are total events, not unique users.

In [13]:
event_count = df['event_type'].value_counts()
event_count

event_type
view        481833
purchase      9758
cart          8409
Name: count, dtype: int64

## Step 2: Unique Users per Event
To build an accurate funnel, we calculate the distinct number of users at each stage. Notice that 'purchase' is higher than 'cart', indicating a direct checkout feature.

In [15]:
distinct_event_count = df.groupby('event_type')['user_id'].nunique()
distinct_event_count

event_type
cart         4441
purchase     7362
view        89108
Name: user_id, dtype: int64

## Step 3: Funnel Stage Deduplication (The Intent Bucket)
Because users can bypass the cart, we create a unified 'Intent' stage (combining 'cart' and 'purchase'). We use `.nunique()` on this subset to ensure users who both carted and purchased are not counted twice.

In [21]:
intent_data = df[df['event_type'].isin(['cart','purchase'])]
true_intent_users = intent_data['user_id'].nunique()
true_intent_users


9048

## Step 4: Dynamic Funnel Aggregation
Finally, assembling these deduplicated metrics into a clean, automated summary table for dashboard ingestion.

In [ ]:
data = {
    "Funnel_Stage": ["01_View", "02_Intent", "03_Purchase"],
    "Unique_Users": [
        distinct_event_count['view'],      
        true_intent_users,                 
        distinct_event_count['purchase']   
    ]
}

funnel_df = pd.DataFrame(data)
funnel_df

,Funnel_Stage,Unique_Users
0,01_View,89108
1,02_Intent,9048
2,03_Purchase,7362


In [37]:
funnel_df['conversion_rate%'] = (funnel_df['Unique_Users'] / funnel_df['Unique_Users'].shift(1)) * 100 
#funnel_df['conversion_rate%'] = funnel_df['Unique_Users'].fillna(100).round(2)  
funnel_df

,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,01_View,89108,NaN,-89008
1,02_Intent,9048,10.153970,-8948
2,03_Purchase,7362,81.366048,-7262
